# SpectraLite — Colab Bootstrap + Phase Launcher

Run this **once per new Colab runtime** (A100 / any GPU), then launch **any** phase notebook.

## Rule for ALL notebooks (Phase 0, 1, 2, …)

**Never double-click** an `.ipynb` in the left **Files** panel.  
That opens a raw text editor (`# %%` view) with **no Run buttons**. This is a Colab UI issue, not a SpectraLite bug — it happens for every notebook.

### Correct way to open ANY phase (keeps same GPU runtime)

1. Finish this Bootstrap (clone + deps + CUDA OK)
2. `File` → `Open notebook` → **GitHub**
3. Repo: `PrabinDevkota/Execution` · Branch: `main`
4. Click the phase file (`Phase0_…`, `Phase1_…`, …)
5. If prompted: connect to the **existing / current runtime** (do not delete runtime)

Or use the **Colab links printed by the launcher cell** below (open in a new tab → connect to this runtime).


### 0. Runtime check

**Runtime → Change runtime type → GPU (A100) → Save**


In [ ]:
import sys
print("In Colab:", "google.colab" in sys.modules)

try:
    import torch
    print("Torch (before bootstrap):", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("WARNING: No CUDA — select GPU runtime first.")
except ImportError:
    print("torch not imported yet")


### 1. Clone or pull full repo

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/PrabinDevkota/Execution.git"
CLONE_ROOT = Path("/content/Execution")
PACKAGE_ROOT = CLONE_ROOT / "SpectraLite"

if not PACKAGE_ROOT.is_dir():
    subprocess.check_call(["git", "clone", REPO_URL, str(CLONE_ROOT)])
else:
    subprocess.check_call(["git", "-C", str(CLONE_ROOT), "pull", "--ff-only"])

if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

%cd {PACKAGE_ROOT}
print("Repo ready:", PACKAGE_ROOT)


### 2. Install Colab-safe dependencies (no torch reinstall)

In [ ]:
from spectralite.colab_setup import ensure_environment

repo_root = ensure_environment(pull=False, install=True, require_cuda_on_colab=True)
print("\nBootstrap complete — safe to open any phase notebook via GitHub.")


### 3. Phase launcher (works for every notebook in `notebooks/`)

Prints one-click Colab URLs for **all** current and future phase notebooks after `git pull`.


In [ ]:
from pathlib import Path

GITHUB_USER = "PrabinDevkota"
GITHUB_REPO = "Execution"
BRANCH = "main"
NOTEBOOKS_DIR = Path("/content/Execution/SpectraLite/notebooks")

print("How to open (ALL phases): File → Open notebook → GitHub → pick file")
print("Or open these links in a NEW TAB, then connect to THIS runtime:\n")

notebooks = sorted(NOTEBOOKS_DIR.glob("*.ipynb"))
if not notebooks:
    print("No notebooks found — did clone/pull succeed?")
else:
    for nb_path in notebooks:
        rel = f"SpectraLite/notebooks/{nb_path.name}"
        url = (
            f"https://colab.research.google.com/github/"
            f"{GITHUB_USER}/{GITHUB_REPO}/blob/{BRANCH}/{rel}"
        )
        print(f"- {nb_path.name}")
        print(f"  {url}\n")


### 4. Optional — Phase 0 quick smoke test in this notebook

Only if you want a fast check without opening `Phase0_Setup.ipynb`.  
For Phase 1+, always use the launcher links / GitHub Open above.


In [ ]:
RUN_PHASE0_HERE = True  # set False to skip

if not RUN_PHASE0_HERE:
    print("Skipped Phase 0 smoke test.")
else:
    from spectralite import default_config, set_seed, gpu
    from spectralite.model_loader import (
        load_model_and_tokenizer,
        print_model_load_summary,
        generate_text,
    )
    from spectralite.model_analysis import print_full_model_analysis
    from spectralite.utils import print_checklist, print_section

    cfg = default_config()
    cfg.ensure_directories()
    set_seed(cfg.seed)

    mem_before = gpu.snapshot(label="Memory before loading")
    model, tokenizer = load_model_and_tokenizer(config=cfg)
    print_model_load_summary(model, model_name=cfg.model_name)
    mem_after_load = gpu.snapshot(label="Memory after loading")

    analysis = print_full_model_analysis(model, model_name=cfg.model_name)
    generated = generate_text(
        model, tokenizer, cfg.smoke_prompt,
        max_new_tokens=cfg.max_new_tokens, do_sample=False,
    )
    print_section("Test Inference")
    print(generated)

    mem_after_infer = gpu.snapshot(label="Memory after inference")
    gpu.print_memory_timeline([mem_before, mem_after_load, mem_after_infer])

    print_checklist(
        [
            ("Environment Ready", True),
            ("GPU Ready", gpu.is_cuda_available()),
            ("Model Loaded Successfully", model is not None),
            ("Inference Successful", bool(generated and generated.strip())),
            (
                "Phase 0 Complete",
                gpu.is_cuda_available() and model is not None and bool(generated.strip()),
            ),
        ]
    )


### Template for every future phase notebook (first cell)

Paste this at the top of Phase1 / Phase2 / … after opening via GitHub:

```python
import sys
from pathlib import Path
root = Path("/content/Execution/SpectraLite")
if root.is_dir() and str(root) not in sys.path:
    sys.path.insert(0, str(root))
from spectralite.colab_setup import ensure_environment
ensure_environment(pull=True, install=False)  # deps already done in Bootstrap
```

If this is a brand-new runtime and you skipped Bootstrap, use `install=True`.
